In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
from astropy.timeseries import LombScargle
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, fbeta_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
def bootstrap_roc_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for ROC-AUC via the percentile method."""
    from sklearn.metrics import roc_auc_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap ROC-AUC")

    point = float(roc_auc_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aucs[i] = point  # fall back to point estimate if degenerate
            continue
        aucs[i] = roc_auc_score(yt, ys)
    lo = float(np.percentile(aucs, 2.5))
    hi = float(np.percentile(aucs, 97.5))
    return point, lo, hi

def bootstrap_pr_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for PR-AUC (average precision) via the percentile method."""
    from sklearn.metrics import average_precision_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap PR-AUC")

    point = float(average_precision_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aps = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aps[i] = point  # fall back to point estimate if degenerate
            continue
        aps[i] = average_precision_score(yt, ys)
    lo = float(np.percentile(aps, 2.5))
    hi = float(np.percentile(aps, 97.5))
    return point, lo, hi
observations = pd.read_pickle('data/observations.pkl')
print(f"observations: {len(observations)}, stars: {observations['star_name'].nunique()}")

observations: 220318, stars: 2026


In [2]:
def compute_star_features(group):
    f = {}
    f['rv_std'] = group['rv_centered'].std()
    f['rv_range'] = group['rv_centered'].max() - group['rv_centered'].min()
    f['rv_mean_abs_dev'] = (group['rv_centered'] - group['rv_centered'].median()).abs().mean()
    f['rv_skew'] = group['rv_centered'].skew()
    f['rv_kurtosis'] = group['rv_centered'].kurtosis()
    f['rv_err_mean'] = group['rv_err'].mean()
    f['rv_err_std'] = group['rv_err'].std()
    f['rhkp_std'] = group['RHKp'].std()
    f['rhkp_range'] = group['RHKp'].max() - group['RHKp'].min()
    f['rhkp_mean'] = group['RHKp'].mean()
    f['halpha_std'] = group['Halpha'].std()
    f['halpha_range'] = group['Halpha'].max() - group['Halpha'].min()
    f['halpha_mean'] = group['Halpha'].mean()
    f['rv_rhkp_corr'] = group['rv_centered'].corr(group['RHKp'])
    f['rv_halpha_corr'] = group['rv_centered'].corr(group['Halpha'])
    f['rhkp_halpha_corr'] = group['RHKp'].corr(group['Halpha'])
    f['rhkp_has_variance'] = 1 if group['RHKp'].std() > 0 else 0
    f['halpha_has_variance'] = 1 if group['Halpha'].std() > 0 else 0
    f['has_exoplanets'] = group['has_exoplanets'].iloc[0]
    return pd.Series(f)

star_features = observations.groupby('star_name').apply(compute_star_features, include_groups=False).reset_index()
star_features = star_features.fillna(0)
star_features['has_exoplanets'] = star_features['has_exoplanets'].astype(int)

physical_features = [
    'rv_std', 'rv_range', 'rv_mean_abs_dev', 'rv_skew', 'rv_kurtosis',
    'rv_err_mean', 'rv_err_std',
    'rhkp_std', 'rhkp_range', 'rhkp_mean',
    'halpha_std', 'halpha_range', 'halpha_mean',
    'rv_rhkp_corr', 'rv_halpha_corr', 'rhkp_halpha_corr',
    'rhkp_has_variance', 'halpha_has_variance',
]

print(f"stars: {len(star_features)}, pos: {star_features['has_exoplanets'].sum()}, neg: {(star_features['has_exoplanets'] == 0).sum()}")

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


stars: 2026, pos: 430, neg: 1596


In [3]:
N_FREQ = 500
F_MIN_DAYS = 1.0
LS_METHOD = 'fast'

periodogram_core = ['rv_ls_peak_power', 'rv_ls_log_peak_period',
                    'rv_ls_mean_power', 'rhkp_ls_peak_power', 'rhkp_rv_power_ratio',
                    'halpha_ls_peak_power', 'halpha_rv_power_ratio', 'min_activity_rv_ratio']
periodogram_multi = ['rv_top2_power_ratio', 'rv_top3_power_ratio',
                     'rv_power_conc_top10pct', 'rhkp_power_conc_top10pct',
                     'halpha_top2_power_ratio']
periodogram_fine = ['rv_power_conc_top5pct', 'rv_power_conc_top1pct', 'rv_periodic_snr']
periodogram_features = periodogram_core + periodogram_multi + periodogram_fine

activity_coupling_features = ['rv_rhkp_corr_abs', 'rv_halpha_corr_abs', 'rv_rhkp_partial_corr',
                              'rv_rhkp_corr_std', 'rhkp_rv_ratio']
uncertainty_features = ['rv_excess_std', 'rv_snr', 'rv_weighted_amplitude']

derived_features = activity_coupling_features + periodogram_features + uncertainty_features

summary_features = [f for f in physical_features
                    if f not in ('halpha_has_variance', 'rhkp_has_variance',
                                 'rv_rhkp_corr', 'rv_halpha_corr')]

def compute_derived_features(group):
    f = {}
    rv     = group['rv_centered'].values.astype(float)
    rhkp   = group['RHKp'].values.astype(float)
    halpha = group['Halpha'].values.astype(float)
    bjd    = group['bjd'].values.astype(float)
    rv_err = group['rv_err'].values.astype(float)
    n = len(rv)

    rv_s   = float(np.std(rv))     if n >= 2 else 0.0
    rv_s   = rv_s if rv_s == rv_s else 0.0
    rhkp_s = float(np.std(rhkp))   if n >= 2 else 0.0
    rhkp_s = rhkp_s if rhkp_s == rhkp_s else 0.0
    ha_s   = float(np.std(halpha)) if n >= 2 else 0.0
    ha_s   = ha_s if ha_s == ha_s else 0.0

    rv_var = rv_s ** 2
    mean_rv_err2 = float(np.mean(rv_err ** 2)) if n > 0 else 0.0
    excess_var = max(rv_var - mean_rv_err2, 0.0)
    f['rv_excess_std']  = float(np.sqrt(excess_var))
    f['rv_snr']         = rv_s / float(np.mean(rv_err)) if n > 0 and float(np.mean(rv_err)) > 0 else 0.0
    f['rv_weighted_amplitude'] = rv_s

    f['rv_rhkp_corr_abs']   = abs(float(np.corrcoef(rv, rhkp)[0, 1])) if (rv_s > 0 and rhkp_s > 0) else 0.0
    f['rv_halpha_corr_abs'] = abs(float(np.corrcoef(rv, halpha)[0, 1])) if (rv_s > 0 and ha_s > 0) else 0.0

    if n > 2 and rv_s > 0 and rhkp_s > 0:
        t = bjd - bjd.mean()
        A = np.column_stack([t, np.ones_like(t)])
        def _resid(y):
            coef = np.linalg.lstsq(A, y, rcond=None)[0]
            return y - (coef[0] * t + coef[1])
        rv_r, rhkp_r = _resid(rv), _resid(rhkp)
        std_rv_r, std_rhkp_r = float(np.std(rv_r)), float(np.std(rhkp_r))
        f['rv_rhkp_partial_corr'] = float(np.corrcoef(rv_r, rhkp_r)[0, 1]) if (std_rv_r > 0 and std_rhkp_r > 0) else 0.0
    else:
        f['rv_rhkp_partial_corr'] = 0.0

    w = min(20, n // 2)
    if w >= 5 and rv_s > 0 and rhkp_s > 0:
        step, corrs = max(1, w // 2), []
        for s in range(0, n - w + 1, step):
            if np.std(rhkp[s:s + w]) > 0 and np.std(rv[s:s + w]) > 0:
                c = float(np.corrcoef(rv[s:s + w], rhkp[s:s + w])[0, 1])
                if c == c and not np.isnan(c):
                    corrs.append(c)
        f['rv_rhkp_corr_std'] = float(np.std(corrs)) if len(corrs) >= 2 else 0.0
    else:
        f['rv_rhkp_corr_std'] = 0.0

    f['rhkp_rv_ratio'] = rhkp_s / rv_s if rv_s > 0 else 0.0

    baseline = float(bjd.max() - bjd.min()) if n > 1 else 0.0
    p_max = max(baseline, 2.0) if baseline > 0 else 2.0
    freqs = np.linspace(1.0 / p_max, 1.0 / F_MIN_DAYS, N_FREQ)
    distinct_t = np.unique(np.round(bjd, 6)).size if n > 0 else 0
    can_run_ls = (n >= 4) and (rv_s > 0) and (baseline > 0) and (distinct_t >= 4)

    for k in periodogram_features:
        f[k] = 0.0

    if can_run_ls:
        try:
            ls_rv = LombScargle(bjd, rv, dy=rv_err, normalization='standard', fit_mean=True)
            p_rv = ls_rv.power(freqs, method=LS_METHOD)
            pi = int(np.argmax(p_rv))
            rv_peak_power = float(p_rv[pi])
            rv_peak_period = float(1.0 / freqs[pi]) if freqs[pi] > 0 else 0.0

            x_fit = 2.0 * np.pi * freqs[pi] * (bjd - bjd.min())
            A_fit = np.column_stack([np.sin(x_fit), np.cos(x_fit)])
            coef = np.linalg.lstsq(A_fit, rv, rcond=None)[0]
            fitted_amp = float(np.sqrt(coef[0]**2 + coef[1]**2))
            f['rv_weighted_amplitude'] = fitted_amp

            mean_err = float(np.mean(rv_err)) if n > 0 else 0.0
            f['rv_periodic_snr'] = fitted_amp / mean_err if mean_err > 0 else 0.0

            ls_rhk = LombScargle(bjd, rhkp, normalization='standard', fit_mean=True)
            ls_hal = LombScargle(bjd, halpha, normalization='standard', fit_mean=True)
            p_rhk = ls_rhk.power(freqs, method=LS_METHOD)
            p_hal = ls_hal.power(freqs, method=LS_METHOD)

            rhkp_pow_at_rv = float(p_rhk[pi])
            halpha_pow_at_rv = float(p_hal[pi])
            rhkp_ratio   = rhkp_pow_at_rv / (rv_peak_power + 1e-9)
            halpha_ratio = halpha_pow_at_rv / (rv_peak_power + 1e-9)

            p_sorted_idx = np.argsort(p_rv)[::-1]
            top2 = float(p_rv[p_sorted_idx[1]] / p_rv[p_sorted_idx[0]]) if len(p_sorted_idx) > 1 and p_rv[p_sorted_idx[0]] > 0 else 0.0
            top3 = float(p_rv[p_sorted_idx[2]] / p_rv[p_sorted_idx[0]]) if len(p_sorted_idx) > 2 and p_rv[p_sorted_idx[0]] > 0 else 0.0

            p_sorted = np.sort(p_rv)
            total_p = max(np.sum(p_rv), 1e-9)
            conc_10 = float(np.sum(p_sorted[-(max(1, len(p_rv)//10)):]) / total_p)
            conc_5  = float(np.sum(p_sorted[-(max(1, len(p_rv)//20)):]) / total_p)
            conc_1  = float(np.sum(p_sorted[-(max(1, len(p_rv)//100)):]) / total_p)

            p_rhk_sorted = np.sort(p_rhk)
            rhk_conc_10 = float(np.sum(p_rhk_sorted[-(max(1, len(p_rhk)//10)):]) / max(np.sum(p_rhk), 1e-9))

            p_hal_sorted_idx = np.argsort(p_hal)[::-1]
            hal_top2 = float(p_hal[p_hal_sorted_idx[1]] / p_hal[p_hal_sorted_idx[0]]) if len(p_hal_sorted_idx) > 1 and p_hal[p_hal_sorted_idx[0]] > 0 else 0.0

            f['rv_ls_peak_power']        = rv_peak_power
            f['rv_ls_log_peak_period']   = float(np.log10(max(rv_peak_period, 1e-9)))
            f['rv_ls_mean_power']        = float(np.mean(p_rv))
            f['rhkp_ls_peak_power']      = float(np.max(p_rhk))
            f['rhkp_rv_power_ratio']     = rhkp_ratio
            f['halpha_ls_peak_power']    = float(np.max(p_hal))
            f['halpha_rv_power_ratio']   = halpha_ratio
            f['min_activity_rv_ratio']   = min(rhkp_ratio, halpha_ratio)
            f['rv_top2_power_ratio']     = top2
            f['rv_top3_power_ratio']     = top3
            f['rv_power_conc_top10pct']  = conc_10
            f['rhkp_power_conc_top10pct'] = rhk_conc_10
            f['halpha_top2_power_ratio'] = hal_top2
            f['rv_power_conc_top5pct']   = conc_5
            f['rv_power_conc_top1pct']   = conc_1
        except Exception:
            pass

    f['has_exoplanets'] = int(group['has_exoplanets'].iloc[0])
    return pd.Series(f)

derived_df = (observations.groupby('star_name', sort=True)
              .apply(compute_derived_features, include_groups=False)
              .reset_index()
              .fillna(0))

features_df = star_features.merge(derived_df[['star_name'] + derived_features],
                                  on='star_name', how='left').fillna(0)

In [4]:
def compute_cadence(group):
    bjd = group['bjd'].values.astype(float)
    if len(bjd) >= 2:
        dbjd = np.diff(np.sort(bjd))
        return float(np.median(dbjd))
    return 0.0

cadence = (observations.groupby('star_name', sort=True)
           .apply(compute_cadence, include_groups=False)
           .reset_index()
           .rename(columns={0: 'cadence_median'})
           .fillna(0))

star_data = features_df.merge(cadence[['star_name', 'cadence_median']],
                              on='star_name', how='left').fillna(0)

feature_columns = summary_features + derived_features + ['cadence_median']

X = star_data[feature_columns].values
y = star_data['has_exoplanets'].values.astype(int)

print(f"features: {len(feature_columns)}, stars: {len(y)}, pos: {y.sum()}, neg: {(1 - y).sum()}")

features: 39, stars: 2026, pos: 430, neg: 1596


In [5]:
PARAM_DISTRIBUTIONS = {
    "C": [
        0.01,
        0.1,
        0.5,
        1.0,
        5.0,
        10.0,
        100.0
    ]
}

N_REPS = 5
N_FOLDS = 5
N_INNER_FOLDS = 3
N_ITER = 15
n_stars = len(y)

all_oof_probs = np.zeros((n_stars, N_REPS))
all_oof_preds = np.zeros((n_stars, N_REPS), dtype=int)
rep_metrics = {'rep': [], 'pr_auc': [], 'roc_auc': [],
               'f1': [], 'f05': [], 'precision': [], 'recall': []}

for rep in range(N_REPS):
    oof_probs = np.zeros(n_stars)
    oof_preds = np.zeros(n_stars, dtype=int)
    seed = 42 + rep
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    for train_idx, test_idx in skf.split(X, y):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test = X[test_idx]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        base_clf = LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=2000, random_state=seed)
        inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=seed)
        search = RandomizedSearchCV(
            base_clf, PARAM_DISTRIBUTIONS, n_iter=N_ITER,
            cv=inner_cv, scoring='average_precision', n_jobs=-1,
            random_state=seed,
        )
        search.fit(X_train_s, y_train)

        inner_probs = cross_val_predict(
            search.best_estimator_, X_train, y_train,
            cv=StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=seed), method='predict_proba', n_jobs=-1,
        )[:, 1]
        vp_in, vr_in, vt_in = precision_recall_curve(y_train, inner_probs)
        vf1_in = 2 * vp_in * vr_in / (vp_in + vr_in + 1e-8)
        fold_thr = float(vt_in[int(np.argmax(vf1_in))]) if len(vt_in) > 0 else 0.5

        test_probs = search.predict_proba(X_test_s)[:, 1]
        oof_probs[test_idx] = test_probs
        oof_preds[test_idx] = (test_probs >= fold_thr).astype(int)

    all_oof_probs[:, rep] = oof_probs
    all_oof_preds[:, rep] = oof_preds

    roc = roc_auc_score(y, oof_probs)
    pr  = average_precision_score(y, oof_probs)
    cm = confusion_matrix(y, oof_preds)
    tn, fp, fn, tp = cm.ravel()
    prc = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = f1_score(y, oof_preds, zero_division=0)
    f05 = fbeta_score(y, oof_preds, beta=0.5, zero_division=0)

    rep_metrics['rep'].append(rep)
    rep_metrics['pr_auc'].append(pr)
    rep_metrics['roc_auc'].append(roc)
    rep_metrics['f1'].append(f1)
    rep_metrics['f05'].append(f05)
    rep_metrics['precision'].append(prc)
    rep_metrics['recall'].append(rec)

    print(f"rep {rep} (seed {seed}): pr_auc={pr:.4f} roc_auc={roc:.4f} f1={f1:.4f} f0.5={f05:.4f} p={prc:.3f} r={rec:.3f}")

rep_df = pd.DataFrame(rep_metrics)

print("\nper-rep pr_auc:")
for _, row in rep_df.iterrows():
    print(f"  rep {int(row['rep'])}: {row['pr_auc']:.4f}")

print(f"\naggregate (n={N_REPS} reps):")
for m in ['pr_auc', 'roc_auc', 'f1', 'f05', 'precision', 'recall']:
    v = rep_df[m].values
    print(f"  {m}: {v.mean():.4f} +/- {v.std(ddof=1):.4f} (min={v.min():.4f}, max={v.max():.4f})")

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 7 is smaller than n_iter=15. Running 7 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown i

rep 0 (seed 42): pr_auc=0.4431 roc_auc=0.7606 f1=0.4850 f0.5=0.4067 p=0.367 r=0.714


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

rep 1 (seed 43): pr_auc=0.4463 roc_auc=0.7632 f1=0.4996 f0.5=0.4219 p=0.382 r=0.721


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

rep 3 (seed 45): pr_auc=0.4555 roc_auc=0.7514 f1=0.4796 f0.5=0.3964 p=0.355 r=0.737


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 7 is smaller than n_iter=15. Running 7 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 7 is smaller than n_iter=15. Running 7 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python

rep 4 (seed 46): pr_auc=0.4463 roc_auc=0.7524 f1=0.4709 f0.5=0.3963 p=0.358 r=0.686

per-rep pr_auc:
  rep 0: 0.4431
  rep 1: 0.4463
  rep 2: 0.4346
  rep 3: 0.4555
  rep 4: 0.4463

aggregate (n=5 reps):
  pr_auc: 0.4452 +/- 0.0075 (min=0.4346, max=0.4555)
  roc_auc: 0.7551 +/- 0.0065 (min=0.7479, max=0.7632)
  f1: 0.4817 +/- 0.0114 (min=0.4709, max=0.4996)
  f05: 0.4045 +/- 0.0106 (min=0.3963, max=0.4219)
  precision: 0.3655 +/- 0.0105 (min=0.3554, max=0.3822)
  recall: 0.7070 +/- 0.0250 (min=0.6767, max=0.7372)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
avg_oof = all_oof_probs.mean(axis=1)
combined_pr = average_precision_score(y, avg_oof)
combined_roc = roc_auc_score(y, avg_oof)
combined_preds = (all_oof_preds.mean(axis=1) >= 0.5).astype(int)
combined_f1   = f1_score(y, combined_preds, zero_division=0)
combined_f05  = fbeta_score(y, combined_preds, beta=0.5, zero_division=0)
cm = confusion_matrix(y, combined_preds)
tn, fp, fn, tp = cm.ravel()

print("combined oof (avg across reps):")
print(f"  pr_auc: {combined_pr:.4f}")
print(f"  roc_auc: {combined_roc:.4f}")
print(f"  f1: {combined_f1:.4f}")
print(f"  f0.5: {combined_f05:.4f}")
print(f"  confusion: TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"  precision: {tp/(tp+fp) if (tp+fp)>0 else 0:.4f} recall: {tp/(tp+fn) if (tp+fn)>0 else 0:.4f}")

pr_point, pr_lo, pr_hi = bootstrap_pr_auc(y, avg_oof)
roc_point, roc_lo, roc_hi = bootstrap_roc_auc(y, avg_oof)
print("\nbootstrap 95% ci (200 resamples):")
print(f"  pr_auc: {pr_point:.4f} [{pr_lo:.4f}, {pr_hi:.4f}]")
print(f"  roc_auc: {roc_point:.4f} [{roc_lo:.4f}, {roc_hi:.4f}]")

combined oof (avg across reps):
  pr_auc: 0.4583
  roc_auc: 0.7610
  f1: 0.4814
  f0.5: 0.4040
  confusion: TN=1067 FP=529 FN=126 TP=304
  precision: 0.3649 recall: 0.7070

bootstrap 95% ci (200 resamples):
  pr_auc: 0.4583 [0.4159, 0.5087]
  roc_auc: 0.7610 [0.7374, 0.7855]
